# 506 — Kong AI Gateway: Anthropic Smoke Test

**Phase 506** wires the app's Anthropic client through Kong's **ai-proxy plugin** — the
actual Kong AI Gateway feature — so all AI traffic is routed and authenticated by Kong
before reaching the Anthropic API.

## What this notebook covers

1. Understanding the three URLs that beginners commonly confuse
2. The difference between ai-proxy, request-transformer, and ai-request-transformer
3. Prerequisites from Phase 505
4. Step-by-step Konnect UI instructions (ai-proxy plugin setup)
5. decK command examples
6. Environment variable inspection (masked)
7. Live smoke tests — gated by `ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true`
8. Troubleshooting guide
9. Rollback steps

## What this notebook does NOT cover

- MCP Gateway routing — that is **Phase 507**
- Kong consumer ACL scopes — coming in Phase 507
- Self-managed or hybrid Kong deployments — we use Konnect Serverless only

---

## Runtime behaviour

| `KONG_AI_GATEWAY_ENABLED` | What the app does |
|---|---|
| `false` (default) | Direct Anthropic SDK — unchanged from before Phase 506 |
| `true` | Routes all Anthropic calls through Kong ai-proxy at `KONG_PROXY_URL/ai` |

Setting `KONG_AI_GATEWAY_ENABLED=false` is all you need to roll back.
The direct Anthropic path always works independently of Kong.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

print("Environment loaded.")

Environment loaded.


---

## Part 1 — Three URLs you must not confuse

Kong beginners almost always mix these up. Read this section carefully.

### 1. Konnect API address

```
https://au.api.konghq.com   (or us / eu / in)
```

- Used by **decK** and the **Konnect REST API** to manage configuration.
- Set this as `KONG_KONNECT_ADDR` in your `.env`.
- **This is NOT where you send application traffic.**

### 2. Serverless proxy URL

```
https://abc1234.au.kong.tech   (unique per gateway, shown in Konnect UI)
```

- This is where your **application sends real API traffic**.
- Set this as `KONG_PROXY_URL` in your `.env`.
- Found in: Konnect → **Gateway Manager** → your Serverless gateway → detail page.
- **This is NOT the Konnect API URL.**

### 3. Upstream Anthropic endpoint

```
https://api.anthropic.com
```

- This is the **upstream provider** that Kong's ai-proxy plugin routes to internally.
- Your application never calls this directly when Kong mode is enabled.
- The ai-proxy plugin injects the Anthropic `x-api-key` before forwarding.

### Summary

```
App  ──►  KONG_PROXY_URL/ai  ──►  api.anthropic.com/v1/messages
          (Serverless proxy)        (Kong ai-proxy routes here internally)

decK ──►  KONG_KONNECT_ADDR         (management only, no app traffic)
```

The most common mistake:
```bash
# WRONG — this is the admin API, not the proxy
KONG_PROXY_URL=https://au.api.konghq.com

# CORRECT — use the Serverless proxy URL from Konnect Gateway Manager
KONG_PROXY_URL=https://abc1234.au.kong.tech
```

---

## Part 2 — Which Kong plugin to use (and why)

If you're wondering whether to use `request-transformer`, `ai-request-transformer`,
or something else for routing to Anthropic — here's the answer:

| Plugin | What it actually does | Use for Phase 506? |
|---|---|---|
| **`ai-proxy`** | Kong AI Gateway plugin. Routes to AI providers (Anthropic, OpenAI, etc.), injects upstream API key, handles versioning. | ✅ Yes — this is the right tool |
| `request-transformer` | Static header/body manipulation. Could inject a key, but it's a generic tool not designed for AI routing. | ❌ Wrong tool |
| `ai-request-transformer` | Uses a **configured LLM to rewrite request content** before forwarding. Has nothing to do with API key injection. | ❌ Unrelated |

Phase 506 uses `ai-proxy` configured with `provider: anthropic` and `llm_format: anthropic`.

With `llm_format: anthropic`, Kong preserves the native Anthropic request format —
the app continues to send `{"model":..., "system":..., "messages":[...]}` exactly as it
does in direct mode. No request body changes needed.

---

## Part 3 — Prerequisites

Before running this notebook you must have completed **Phase 505**:

- [ ] decK installed and working (`deck version` prints a version number)
- [ ] Konnect Personal Access Token (PAT) created
- [ ] `entity-risk-ai` Serverless control plane created in Konnect
- [ ] `deck gateway ping` succeeded
- [ ] `KONG_KONNECT_ADDR`, `KONG_KONNECT_TOKEN`, `KONG_KONNECT_CONTROL_PLANE_NAME` set in `.env`

For Phase 506 you additionally need:

- [ ] The **Serverless proxy URL** for your gateway (from Konnect Gateway Manager)
- [ ] A Kong consumer key for the app (you will create this below)
- [ ] These vars set in `.env`:
  - `KONG_PROXY_URL=https://<your-proxy>.au.kong.tech`
  - `KONG_AI_GATEWAY_ENABLED=true`
  - `KONG_AI_GATEWAY_ROUTE_PATH=/ai`
  - `KONG_AI_GATEWAY_API_KEY=<your-consumer-key>`
  - `ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true`  (only if you want live cells to run)

---

## Part 4 — Konnect UI: step-by-step setup

Follow these steps in [cloud.konghq.com](https://cloud.konghq.com).

---

### Step 1 — Find your Serverless gateway and proxy URL

1. Sign in to [cloud.konghq.com](https://cloud.konghq.com)
2. Click **Gateway Manager** in the left sidebar
3. Find your gateway named **entity-risk-ai** (or **serverless-default** if using the default)
4. Click the gateway name to open its detail page
5. Look for the **Proxy URL** — it looks like `https://abc1234.au.kong.tech`
6. Copy this and set it as `KONG_PROXY_URL` in your `.env`

---

### Step 2 — Create a Service (placeholder upstream)

The Service URL is a **placeholder** — the ai-proxy plugin overrides the upstream entirely.
Kong never connects to this address.

1. Inside your gateway, click **Services** → **+ New Service**
2. Fill in:
   - **Name:** `anthropic-ai`
   - **Upstream URL:** `http://localhost:32000`
     *(placeholder — ai-proxy overrides this. Any non-routable value works.)*
3. Click **Save**

---

### Step 3 — Add a route `/ai` to the service

1. Click the **anthropic-ai** service
2. Click **Routes** → **+ New Route**
3. Fill in:
   - **Name:** `ai-route`
   - **Paths:** `/ai`
   - **Methods:** `POST`
   - **Strip Path:** ❌ Disabled
     *(ai-proxy handles the upstream path internally — do NOT strip the prefix)*
4. Click **Save**

> **Why strip_path off?**  
> The ai-proxy plugin intercepts the request at the route and routes to
> `api.anthropic.com/v1/messages` on its own. If strip_path were enabled, Kong
> would modify the path before the plugin sees it, which can cause issues.

---

### Step 4 — Add the ai-proxy plugin (Kong AI Gateway)

This is the core step that makes this an AI Gateway route.

1. Click the **ai-route** route
2. Click **Plugins** → **+ Add Plugin**
3. Search for **AI Proxy** and click **Enable**
4. Configure:
   - **Route Type:** `llm/v1/chat`
   - **LLM Format:** `anthropic`
     *(preserves native Anthropic request format — no translation needed)*
   - **Auth → Header Name:** `x-api-key`
   - **Auth → Header Value:** `<your-real-anthropic-api-key>`
     *(this is the Anthropic credential — the app does NOT send it)*
   - **Model → Provider:** `anthropic`
   - **Model → Name:** `claude-haiku-4-5-20251001`
   - **Model Options → anthropic_version:** `2023-06-01`
   - **Model Options → max_tokens:** `1024`
5. Click **Save**

> **Security note:** The Anthropic API key is now stored in Konnect (in the ai-proxy
> plugin config), not in your app. In Kong mode, only `KONG_AI_GATEWAY_API_KEY`
> (your consumer key) is needed by the app.

---

### Step 5 — Add the key-auth plugin to the route

1. Still on **ai-route**, click **Plugins** → **+ Add Plugin**
2. Search for **Key Authentication** and click **Enable**
3. Configure:
   - **Key Names:** `x-kong-api-key`, `X-Kong-API-Key`
   - **Hide Credentials:** ✅ Enabled
4. Click **Save**

Requests without a valid `X-Kong-API-Key` now receive HTTP 401.

---

### Step 6 — Add the rate-limiting plugin to the route

1. Still on **ai-route**, click **Plugins** → **+ Add Plugin**
2. Search for **Rate Limiting** and click **Enable**
3. Configure:
   - **Minute:** `20`
   - **Policy:** `local`
4. Click **Save**

---

### Step 7 — Create a Consumer and API key credential

1. In the left menu, click **Consumers** → **+ New Consumer**
2. Fill in:
   - **Username:** `entity-risk-ai-app`
3. Click **Save**
4. Click the consumer → **Credentials** → **Key Auth** → **+ New Key Auth**
5. Enter a key value (or let Kong generate one):
   ```bash
   # Generate a strong key
   python3 -c "import secrets; print(secrets.token_urlsafe(32))"
   ```
6. Copy the key and set `KONG_AI_GATEWAY_API_KEY=<the-key>` in your `.env`

---

### Step 8 — Verify the proxy URL

Check `KONG_PROXY_URL` in your `.env`:

- ✅ Correct: `https://abc1234.au.kong.tech`
- ❌ Wrong (admin API): `https://au.api.konghq.com`

Quick connectivity test (expects 401 — confirms route exists and key-auth is active):

```bash
curl -s -o /dev/null -w "%{http_code}" \
  -X POST "$KONG_PROXY_URL/ai" \
  -H "Content-Type: application/json"
# Should print: 401
```

---

## Part 5 — decK examples

If you prefer managing config via decK instead of the UI:

```bash
# Set the Anthropic key as an env var (use decK env var substitution)
export DECK_ANTHROPIC_API_KEY="sk-ant-..."

# Preview what decK would change (dry run)
deck gateway diff \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  kong/declarative/phase-506-ai-route.yaml

# Apply the config
deck gateway sync \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  kong/declarative/phase-506-ai-route.yaml

# Dump the live config to a file
deck gateway dump \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  --output-file kong/declarative/current-live-state.yaml
```

The YAML at `kong/declarative/phase-506-ai-route.yaml` uses
`${{ env "DECK_ANTHROPIC_API_KEY" }}` for the ai-proxy `auth.header_value`,
so the key is substituted at sync time and never stored in the file.

> **Note:** `--konnect-region` is a deprecated flag. Always use `--konnect-addr`.

---

## Part 6 — Environment inspection

In [3]:
from src.config import get_kong_settings, get_kong_ai_gateway_settings

kong = get_kong_settings()
ai_gw = get_kong_ai_gateway_settings()

print("Phase 505 — Konnect control plane settings:")
for k, v in kong.masked().items():
    status = "✅" if v else "⚠️  (not set)"
    print(f"  {k:<35} {v or '(empty)'}  {status}")

print()
print("Phase 506 — Kong AI Gateway settings:")
for k, v in ai_gw.masked().items():
    if k == "enabled":
        status = "✅ Kong mode ACTIVE" if v else "ℹ️  direct Anthropic mode"
    else:
        status = "✅" if (v and v != "(proxy_url not set)") else "⚠️  (not set)"
    print(f"  {k:<35} {v}  {status}")

Phase 505 — Konnect control plane settings:
  konnect_region                      au  ✅
  konnect_addr                        https://au.api.konghq.com  ✅
  konnect_control_plane_name          entity-risk-ai  ✅
  konnect_token                       kpat_LuC…***  ✅
  konnect_control_plane_id            a92e7da1-469b-4cda-851a-eb326907df7c  ✅
  notebook_live_tests                 True  ✅

Phase 506 — Kong AI Gateway settings:
  enabled                             True  ✅ Kong mode ACTIVE
  proxy_url                           https://kong-948ef205bdausa0ym.kongcloud.dev  ✅
  route_path                          /ai  ✅
  api_key                             KRZLzUZy…***  ✅
  gateway_url                         https://kong-948ef205bdausa0ym.kongcloud.dev/ai  ✅


In [4]:
LIVE = os.getenv("ENABLE_LIVE_KONG_NOTEBOOK_TESTS", "").strip().lower() == "true"
KONG_ENABLED = os.getenv("KONG_AI_GATEWAY_ENABLED", "").strip().lower() == "true"

print(f"ENABLE_LIVE_KONG_NOTEBOOK_TESTS : {LIVE}")
print(f"KONG_AI_GATEWAY_ENABLED         : {KONG_ENABLED}")

if not LIVE:
    print()
    print("⏭  Live test cells are skipped.")
    print("   Set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true in .env to run them.")
elif not KONG_ENABLED:
    print()
    print("ℹ️  KONG_AI_GATEWAY_ENABLED=false — live tests will use direct Anthropic mode.")
    print("   Set KONG_AI_GATEWAY_ENABLED=true to test the Kong ai-proxy path.")
else:
    print()
    print("✅ Kong AI Gateway mode enabled. Live cells will hit the real gateway.")

ENABLE_LIVE_KONG_NOTEBOOK_TESTS : True
KONG_AI_GATEWAY_ENABLED         : True

✅ Kong AI Gateway mode enabled. Live cells will hit the real gateway.


---

## Part 7 — Route URL preview

In [5]:
from src.config import get_kong_ai_gateway_settings

s = get_kong_ai_gateway_settings()

print("URL breakdown for Kong AI Gateway mode:")
print(f"  KONG_PROXY_URL           : {s.proxy_url or '(not set)'}")
print(f"  KONG_AI_GATEWAY_ROUTE    : {s.route_path}")
print(f"  gateway_url              : {s.gateway_url if s.proxy_url else '(proxy_url not set)'}")
print()
print("What the app sends (Kong mode):")
print(f"  POST {s.gateway_url if s.proxy_url else '(proxy_url not set)'}")
print(f"  Header: X-Kong-API-Key: {'<set>' if s.api_key else '(not set)'}")
print(f"  Body: {{model: ..., system: ..., messages: [...]}}  (native Anthropic format)")
print()
print("What Kong ai-proxy does internally:")
print("  POST https://api.anthropic.com/v1/messages")
print("  Header: x-api-key: <from ai-proxy auth.header_value in Konnect>")
print("  Header: anthropic-version: 2023-06-01  <from ai-proxy model.options>")


URL breakdown for Kong AI Gateway mode:
  KONG_PROXY_URL           : https://kong-948ef205bdausa0ym.kongcloud.dev
  KONG_AI_GATEWAY_ROUTE    : /ai
  gateway_url              : https://kong-948ef205bdausa0ym.kongcloud.dev/ai

What the app sends (Kong mode):
  POST https://kong-948ef205bdausa0ym.kongcloud.dev/ai
  Header: X-Kong-API-Key: <set>
  Body: {model: ..., system: ..., messages: [...]}  (native Anthropic format)

What Kong ai-proxy does internally:
  POST https://api.anthropic.com/v1/messages
  Header: x-api-key: <from ai-proxy auth.header_value in Konnect>
  Header: anthropic-version: 2023-06-01  <from ai-proxy model.options>


---

## Part 8 — Live smoke tests

In [6]:
# ── Smoke test 1: HTTP connectivity check ──────────────────────────────────
#
# Sends a request with no API key to the Kong /ai route.
# Expects HTTP 401 from key-auth — confirms the route exists and key-auth is active.

if not LIVE:
    print("⏭  Skipped (ENABLE_LIVE_KONG_NOTEBOOK_TESTS=false)")
else:
    s = get_kong_ai_gateway_settings()
    if not s.proxy_url:
        print("❌ FAIL: KONG_PROXY_URL is not set.")
    else:
        url = s.gateway_url
        print(f"Sending no-key request to: {url}")
        try:
            resp = requests.post(
                url,
                headers={"Content-Type": "application/json"},
                json={"test": True},
                timeout=15,
            )
            print(f"HTTP status: {resp.status_code}")
            if resp.status_code == 401:
                print("✅ PASS — HTTP 401 as expected (key-auth is active, route exists)")
            elif resp.status_code == 404:
                print("❌ FAIL — HTTP 404: /ai route not found.")
                print("   Check: does the ai-route exist in Konnect with path /ai?")
                print(f"   Response: {resp.text[:300]}")
            elif resp.status_code == 200:
                print("⚠️  HTTP 200 without a key — key-auth may not be active on this route.")
            else:
                print(f"⚠️  Unexpected HTTP {resp.status_code}: {resp.text[:300]}")
        except requests.exceptions.ConnectionError as e:
            print(f"❌ FAIL — Cannot connect to {url}")
            print("   Is KONG_PROXY_URL the Serverless proxy URL (not the Konnect API URL)?")
            print(f"   Error: {e}")

Sending no-key request to: https://kong-948ef205bdausa0ym.kongcloud.dev/ai
HTTP status: 401
✅ PASS — HTTP 401 as expected (key-auth is active, route exists)


In [7]:
# ── Smoke test 2: Authenticated route + ai-proxy reachability ──────────────
#
# Sends a valid X-Kong-API-Key with a minimal invalid payload.
# Expects HTTP 400 (Anthropic rejects the payload) — confirms key-auth passes
# and ai-proxy is forwarding to Anthropic successfully.

if not LIVE:
    print("⏭  Skipped (ENABLE_LIVE_KONG_NOTEBOOK_TESTS=false)")
else:
    s = get_kong_ai_gateway_settings()
    if not s.proxy_url:
        print("❌ FAIL: KONG_PROXY_URL is not set.")
    elif not s.api_key:
        print("❌ FAIL: KONG_AI_GATEWAY_API_KEY is not set.")
    else:
        url = s.gateway_url
        print(f"Sending authenticated request to: {url}")
        try:
            resp = requests.post(
                url,
                headers={
                    "Content-Type": "application/json",
                    "X-Kong-API-Key": s.api_key,
                },
                json={"invalid": "payload"},  # intentionally bad
                timeout=15,
            )
            print(f"HTTP status: {resp.status_code}")
            if resp.status_code in (400, 422):
                print("✅ PASS — key-auth passed, ai-proxy forwarded to Anthropic")
                print("          (Anthropic rejected the bad payload — that's expected)")
            elif resp.status_code == 401:
                print("❌ FAIL — HTTP 401: Kong rejected the API key.")
                print("   Check KONG_AI_GATEWAY_API_KEY matches the consumer credential.")
                print(f"   Response: {resp.text[:300]}")
            elif resp.status_code == 403:
                print("❌ FAIL — HTTP 403: Key valid but consumer lacks permission.")
                print(f"   Response: {resp.text[:300]}")
            elif resp.status_code == 429:
                print("⚠️  HTTP 429: Rate limit exceeded. Wait 1 minute and retry.")
            elif resp.status_code >= 500:
                print(f"❌ FAIL — HTTP {resp.status_code}: ai-proxy or Anthropic error.")
                print("   Check: ai-proxy plugin is on the route, auth.header_value is set.")
                print(f"   Response: {resp.text[:400]}")
            else:
                print(f"ℹ️  HTTP {resp.status_code}: {resp.text[:300]}")
        except requests.exceptions.ConnectionError as e:
            print(f"❌ FAIL — Cannot connect: {e}")

Sending authenticated request to: https://kong-948ef205bdausa0ym.kongcloud.dev/ai
HTTP status: 400
✅ PASS — key-auth passed, ai-proxy forwarded to Anthropic
          (Anthropic rejected the bad payload — that's expected)


In [8]:
# ── Smoke test 3: Full model invocation via Kong ai-proxy ──────────────────
#
# Uses AnthropicClient in Kong mode to make a real model call end-to-end.
# Verifies: app → Kong key-auth → ai-proxy → Anthropic → response.

if not LIVE:
    print("⏭  Skipped (ENABLE_LIVE_KONG_NOTEBOOK_TESTS=false)")
elif not KONG_ENABLED:
    print("⏭  Skipped (KONG_AI_GATEWAY_ENABLED=false — set to true to test Kong path)")
else:
    from src.config import get_anthropic_settings, get_kong_ai_gateway_settings
    from src.clients.anthropic_client import AnthropicClient

    anthropic_settings = get_anthropic_settings()
    kong_settings = get_kong_ai_gateway_settings()

    print(f"Kong gateway URL : {kong_settings.gateway_url}")
    print(f"Model            : {anthropic_settings.model_haiku}")
    print()

    client = AnthropicClient(
        settings=anthropic_settings,
        kong_settings=kong_settings,
    )

    try:
        result = client.generate_text(
            system_prompt="You are a concise assistant.",
            user_prompt="Reply with exactly one word: the colour of the sky.",
            model=anthropic_settings.model_haiku,
            max_tokens=20,
        )
        print(f"Response : {result!r}")
        print(f"Usage    : {client.last_usage}")
        print()
        if result and client.last_usage and client.last_usage.get("total_tokens", 0) > 0:
            print("✅ PASS — Model invocation via Kong AI Gateway succeeded")
        else:
            print("⚠️  Response received but usage tracking may be empty")
    except RuntimeError as e:
        print(f"❌ FAIL — {e}")

Kong gateway URL : https://kong-948ef205bdausa0ym.kongcloud.dev/ai
Model            : claude-haiku-4-5-20251001

Response : 'Blue.'
Usage    : {'input_tokens': 19, 'output_tokens': 5, 'total_tokens': 24, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0}

✅ PASS — Model invocation via Kong AI Gateway succeeded


In [9]:
# ── Smoke test 4: Direct mode still works (regression check) ───────────────
#
# Confirms the direct Anthropic path is unaffected by Phase 506.
# Runs regardless of KONG_AI_GATEWAY_ENABLED.

if not LIVE:
    print("⏭  Skipped (ENABLE_LIVE_KONG_NOTEBOOK_TESTS=false)")
else:
    from src.config import get_anthropic_settings
    from src.clients.anthropic_client import AnthropicClient

    anthropic_settings = get_anthropic_settings()
    direct_client = AnthropicClient(settings=anthropic_settings, kong_settings=None)

    print("Testing direct Anthropic mode (kong_settings=None)...")
    try:
        result = direct_client.generate_text(
            system_prompt="You are a concise assistant.",
            user_prompt="Reply with exactly one word: the colour of grass.",
            model=anthropic_settings.model_haiku,
            max_tokens=20,
        )
        print(f"Response : {result!r}")
        print(f"Usage    : {direct_client.last_usage}")
        print()
        print("✅ PASS — Direct Anthropic path still works (no regression)")
    except RuntimeError as e:
        print(f"❌ FAIL — Direct mode error: {e}")

Testing direct Anthropic mode (kong_settings=None)...
Response : 'Green'
Usage    : {'input_tokens': 25, 'output_tokens': 4, 'total_tokens': 29, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0}

✅ PASS — Direct Anthropic path still works (no regression)


---

## Part 9 — Troubleshooting

### HTTP 401 — Kong rejected the API key

- Check `KONG_AI_GATEWAY_API_KEY` in `.env` matches the key in Konnect → Consumers → entity-risk-ai-app → Credentials → Key Auth.
- Check the key-auth plugin has `x-kong-api-key` and/or `X-Kong-API-Key` in the Key Names list.
- Reload `.env`: run `load_dotenv(override=True)` and re-run the setup cell.

### HTTP 403 — Forbidden

- Key is valid but the consumer does not have permission. Check for unexpected ACL plugins.
- Phase 506 does not use ACL — if you see 403, look for extra plugins on the route.

### HTTP 404 — Route not found

- The `/ai` route does not exist in Konnect.
- Verify: Konnect → Gateway Manager → your gateway → Services → anthropic-ai → Routes → ai-route.
- Check `KONG_AI_GATEWAY_ROUTE_PATH=/ai` in `.env`.

### HTTP 429 — Rate limit exceeded

- Wait 1 minute and retry (default limit: 20 req/min).
- To raise the limit: Konnect → ai-route → Plugins → Rate Limiting → edit the `minute` value.

### HTTP 500 / 502 — Gateway or upstream error

- Check the **ai-proxy plugin** is enabled on the `ai-route` route.
- Verify `auth.header_value` in the ai-proxy plugin config contains the real Anthropic API key.
- Verify `model.provider: anthropic` and `llm_format: anthropic` are set.
- If the Anthropic key is wrong, Anthropic returns 401 which Kong may surface as 5xx.

### Connection error / cannot reach proxy

- `KONG_PROXY_URL` is almost certainly set to the wrong URL.
- Correct format: `https://abc1234.au.kong.tech`
- **Not** `https://au.api.konghq.com` (that is the management URL).

### Kong mode is never used despite KONG_AI_GATEWAY_ENABLED=true

- Check the value is exactly `true` (not `True` or `1`).
- Reload `.env`: `load_dotenv(override=True)` and restart the notebook kernel.
- In the Streamlit app, restart the process: `streamlit run app.py`.

### "I thought I needed request-transformer or ai-request-transformer"

- You do not. The `ai-proxy` plugin handles upstream auth and routing natively.
- `request-transformer` = generic header edits (not AI Gateway).
- `ai-request-transformer` = uses an LLM to rewrite request content (completely different use case).

---

## Part 10 — Rollback

Phase 506 is **default-safe**. Rolling back means turning off Kong mode.

### Step 1 — Disable Kong mode in `.env`

```bash
KONG_AI_GATEWAY_ENABLED=false
```

That is all that is required. The app reverts to direct Anthropic SDK.

### Step 2 — Restart the Streamlit app (if running)

```bash
streamlit run app.py
```

### Step 3 — Confirm rollback

In [ ]:
load_dotenv(dotenv_path="../.env", override=True)
from src.config import get_kong_ai_gateway_settings
s = get_kong_ai_gateway_settings()

print(f"KONG_AI_GATEWAY_ENABLED = {s.enabled}")
if not s.enabled:
    print("✅ Kong mode is OFF — app uses direct Anthropic path")
else:
    print("ℹ️  Kong mode is still ON — set KONG_AI_GATEWAY_ENABLED=false to roll back")

### Optional: remove the Kong route

```bash
# Option A — delete via Konnect UI
# Gateway Manager → anthropic-ai service → Routes → ai-route → Delete

# Option B — reset to empty state via decK
deck gateway sync \
  --konnect-addr "$KONG_KONNECT_ADDR" \
  --konnect-token "$KONG_KONNECT_TOKEN" \
  --konnect-control-plane-name "$KONG_KONNECT_CONTROL_PLANE_NAME" \
  kong-empty-state.yaml
```

---

## Summary

| What | Detail |
|---|---|
| Default app behaviour | Direct Anthropic SDK — unchanged from before Phase 506 |
| Kong mode activated by | `KONG_AI_GATEWAY_ENABLED=true` |
| Kong plugin used | `ai-proxy` (Kong AI Gateway feature) |
| App authenticates to Kong | `X-Kong-API-Key` (from `KONG_AI_GATEWAY_API_KEY`) |
| Kong authenticates to Anthropic | `x-api-key` from `ai-proxy auth.header_value` in Konnect |
| Request format | Native Anthropic (`llm_format: anthropic`) — no translation |
| Rate limiting | 20 req/min (tune in Konnect) |
| Rollback | `KONG_AI_GATEWAY_ENABLED=false` + restart app |
| Phase 507 | MCP Gateway routing — not in this phase |